# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

In [ ]:
%%html
<style>
    table {
        display: inline-block
    }
</style>

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory","24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Claro/home")
_files_claro = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*SS8BF/*.gz") if file.is_file()]
df = pd.DataFrame(_files_claro, columns=["file_path","parent_folder"])
df["exchange_id"] = df["parent_folder"].str.slice(stop=4)
files_claro = df.groupby("exchange_id").sample(5,random_state=42,replace=True)["file_path"].drop_duplicates().to_list()
files_claro[:5]

In [ ]:
len(files_claro)*6

### Arquivos intermediário e final

In [ ]:
extracted_claro ="/assets/data/cdr/teletools/db/extraidos/ss8bf_claro_extraido.parquet"
transformed_claro ="/assets/data/cdr/teletools/db/transformados/ss8bf_claro_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
from teleutils.core.extractors.schemas._claro import stfc_ss8bf_claro_schema

In [ ]:
df_raw = spark.read.csv(files_claro, sep=",", schema=stfc_ss8bf_claro_schema)
df_raw = df_raw.filter(F.col("record_type").isNotNull())
df_raw.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_raw.groupBy("record_type").count().show()

In [ ]:
(df_raw
 .withColumn(
 "FORW", 
 F.when(F.col("forwarded_from_dn").isNotNull(),1).otherwise(0))
 .groupBy(["originating_party_identifier"])
 .pivot("terminating_party_identifier")
 .count()
).show()

In [ ]:
(df_raw
 .withColumn(
 "FORW", 
 F.when(F.col("forwarded_from_dn").isNotNull(),1).otherwise(0))
 .filter(F.col("originating_number_calling_party").isNotNull())
 .groupBy(["originating_party_identifier"])
 .pivot("terminating_party_identifier")
 .count()
).show()

In [ ]:
df_raw.filter(F.col("originating_party_identifier").isNull()).filter(F.col("originating_number_calling_party").isNotNull()).limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_raw.filter(F.col("originating_party_identifier")=="901").filter(F.col("terminating_party_identifier")=="999").limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_raw.filter(F.col("originating_party_identifier")=="901").filter(F.col("terminating_party_identifier")=="999").groupBy("duration_of_call").count().show()

In [ ]:
df_raw.filter(F.col("duration_of_call")==0).groupBy("originating_party_identifier").count().show()

In [ ]:
column_indices = (2,3,4,5,10,11,17,63,125,126)
column_names = ("data_hora", "duracao", "bilhetador", "referencia", "numero_destino", "numero_origem", "resultado_chamada", "_sigame", "rota_entrada", "rota_saida")
columns_to_keep = [
 F.col(df_raw.columns[index]).alias(column_name)
 for index, column_name in zip(
 column_indices,
 column_names,
 )
]
df = df_raw.select(*columns_to_keep).withColumn("duracao", F.col("duracao").cast(T.IntegerType()))
df = df.filter(F.col("data_hora").isNotNull())

df.show(5, truncate=False)

In [ ]:
df.groupBy("resultado_chamada").count().show()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_claro, extracted_claro,"stfc_ss8bf_claro")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5, truncate=False)

In [ ]:
df_extracted.limit(5).toPandas

In [ ]:
# _referencia
# O campo 6 (Record ID) é um número que identifica exclusivamente a chamada no hiQ 8000 composto de 32 characteres. 
# O SAT-R utiliza os últimos 8 characters que são seqüenciais e baseados na data/hora de início da chamada. 

In [ ]:
## _resultado_chamada
# Specifies the result of the call attempt:
# 0=Call was completed.
# 1=Call was not completed due to called party busy.
# 2=Call was not completed due to invalid dialed number.
# 3=Call was not completed due to lack of available lines/trunks to complete the call.
# 4=Call was not completed due to calling party aborting the call prior to answer.
# 5=Call was not completed due to called party not answering the call.
# 6=Call was not completed due to a network problem. 
# 7=Call was not completed due to unknown reasons. 
# 8=Call was not completed due to no subscriber account.
# 9=Call was not completed due to unauthorized subscriber.
# Default value is no character (NULL).
df_extracted.groupBy("_resultado_chamada").count().orderBy("_resultado_chamada").show()

In [ ]:
# _identificador_origem (Originating Party Identifier)
# 900=Originating. If calling party belongs to hiQ
# 901=Incoming. If calling party does not belong to hiQ

# _identificador_destino (Terminating Party Identifier)
# 902=Terminating. If called party belongs to hiQ
# 903=Outgoing. If called party does not belong to hiQ
# 999=Unknown

# _chamada_encaminhada
# true se `Forwarded from DN` não for nulo, caso contrário, false

# _chamada_encaminhada = false
#900/902 → Originating
#900/903 → Originating
#900/999 → Originating
#901/902 → Terminating
#901/903 → Transit
#901/999 → Unknown

# _chamada_encaminhada = true
#900/902 → Originating
#900/903 → Originating
#900/999 → Originating
#901/902 → Forwarding
#901/903 → Forwarding
#901/999 → Unknown


df_extracted.filter(F.col("_identificador_origem").isNotNull()).withColumn("_chamada_encaminhada",F.col("_numero_encaminhado").isNotNull()).groupBy(["_chamada_encaminhada","_identificador_origem", "_identificador_destino"]).count().orderBy("_chamada_encaminhada","_identificador_origem","_identificador_destino").show()

| _chamada_encaminhada | _identificador_origem | _identificador_destino | resultado |
| :--  | :-- | :-- | :--         |
| false| 900 | 902 | Originating |
| false| 900 | 903 | Originating |
| false| 900 | 999 | Originating |
| false| 901 | 902 |Terminating  |
| false| 901 | 903 |Transit      |
| false| 901 | 999 | Unknown     |
| true | 900 | 902 | Originating |
| true | 900 | 903 | Originating |
| true | 900 | 999 | Originating |
| true | 901 | 902 | Forwarding  |
| true | 901 | 903 | Forwarding  |
| true | 901 | 999 | Unknown     |


## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_ss8bf_claro(extracted_claro, transformed_claro)

In [ ]:
df_transformed = spark.read.parquet(transformed_claro)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_transformed.groupBy("no_tipo_chamada").count().show()